# Avance preliminar: DroneFace + ArcFace

Objetivo experimental: construir y documentar un baseline reproducible de reconocimiento facial closed-set sobre DroneFace usando InsightFace/ArcFace preentrenado. No se entrena una red desde cero, no se usa SCface y no se implementa control de dron.


## 2. Librerias y configuracion


In [ ]:
from pathlib import Path
import os
import re

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.config import (
    PROJECT_ROOT, DATA_DIR, DRONEFACE_ROOT, FIGS_DIR, OUTPUTS_DIR,
    INDEX_DIR, EMBEDDINGS_DIR, RESULTS_DIR, TABLES_DIR, ensure_project_dirs,
)
from src.dataset_index import find_images, inspect_droneface_dataset, build_droneface_index
from src.face_model import FaceRecognizerArcFace
from src.evaluation import (
    build_gallery,
    evaluate_closed_set,
    load_gallery_npz,
    compute_global_metrics,
    compute_grouped_metrics,
    generate_preliminary_discussion,
)
from src.visualization import (
    plot_dataset_samples_droneface,
    plot_detection_examples,
    plot_recognition_examples,
    plot_metric_bar,
    plot_detection_vs_accuracy,
    plot_accuracy_heatmap_height_distance,
    plot_confusion_matrix,
)

ensure_project_dirs()
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DRONEFACE_ROOT: {DRONEFACE_ROOT}")


## 3. Exploracion de DroneFace


In [ ]:
drone_info = inspect_droneface_dataset(DRONEFACE_ROOT, n_examples=30)
drone_paths = find_images(DRONEFACE_ROOT)
print(f"Total DroneFace images: {len(drone_paths)}")
if not drone_paths:
    raise RuntimeError("No DroneFace images found. Check data/DroneFace/open_data_set.")


## 4. Construccion del indice de DroneFace


In [ ]:
drone_df = build_droneface_index(DRONEFACE_ROOT, save=True)
index_path = INDEX_DIR / "droneface_index.csv"
drone_df = pd.read_csv(index_path)

display(drone_df.head())
display(drone_df["source_folder"].value_counts(dropna=False).rename_axis("source_folder").reset_index(name="n"))
display(drone_df["image_type"].value_counts(dropna=False).rename_axis("image_type").reset_index(name="n"))
display(drone_df["subject_id"].value_counts(dropna=False).rename_axis("subject_id").reset_index(name="n"))
display(drone_df["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="n"))
display(
    drone_df[drone_df["split"] == "probe"]
    .groupby(["height_m", "distance_m"], dropna=False)
    .size()
    .reset_index(name="n_probes")
    .sort_values(["height_m", "distance_m"])
)


## 5. Visualizacion inicial de ejemplos


In [ ]:
dataset_fig_path = FIGS_DIR / "dataset" / "droneface_samples.png"
plot_dataset_samples_droneface(drone_df, dataset_fig_path)
print(f"Figura generada: {dataset_fig_path}")


## 6. Inicializacion del modelo ArcFace


In [ ]:
try:
    model = FaceRecognizerArcFace(model_name="buffalo_l", allow_cuda=False)
except Exception as exc:
    print(f"ERROR initializing InsightFace/ArcFace: {type(exc).__name__}: {exc}")
    raise

print(f"Model: {model.model_name}")
print(f"Providers: {model.providers}")


## 7. Ejemplos de deteccion facial


In [ ]:
detections_fig_path = FIGS_DIR / "detection" / "droneface_detections.png"
plot_detection_examples(drone_df, model, detections_fig_path, max_images=8)
print(f"Figura generada: {detections_fig_path}")


## 8. Construccion de galeria


In [ ]:
gallery_df = drone_df[drone_df["split"] == "gallery"].copy()
probe_df = drone_df[drone_df["split"] == "probe"].copy()
gallery_subjects_before = set(gallery_df["subject_id"].dropna().astype(str))
valid_probe_before = probe_df[probe_df["subject_id"].astype(str).isin(gallery_subjects_before)].copy()

print(f"Gallery rows: {len(gallery_df)}")
print(f"Probe rows: {len(probe_df)}")
print(f"Gallery subjects before detection: {len(gallery_subjects_before)}")
print(f"Probes excluded because subject has no gallery row: {len(probe_df) - len(valid_probe_before)}")
print(f"Probes eligible before detection: {len(valid_probe_before)}")

if gallery_df.empty:
    raise RuntimeError("No gallery rows found in droneface_index.csv.")
if probe_df.empty:
    raise RuntimeError("No probe rows found in droneface_index.csv.")
if valid_probe_before.empty:
    raise RuntimeError("No probes have subjects present in gallery.")

gallery_embeddings, gallery_info_df, gallery_failures_df = build_gallery(
    drone_df,
    model,
    dataset_name="DroneFace",
    save_path=EMBEDDINGS_DIR / "droneface_gallery_embeddings.npz",
    failures_path=TABLES_DIR / "gallery_failures_droneface.csv",
)

display(gallery_info_df)
print(f"Gallery subjects with valid embeddings: {len(gallery_embeddings)}")
if not gallery_embeddings:
    raise RuntimeError("No valid gallery embeddings were created.")


## 9. Evaluacion closed-set


In [ ]:
gallery_embedding_subjects = set(gallery_embeddings.keys())
probes_with_gallery_embedding = probe_df[probe_df["subject_id"].astype(str).isin(gallery_embedding_subjects)].copy()
print(f"Probes excluded because subject has no valid gallery embedding: {len(probe_df) - len(probes_with_gallery_embedding)}")
print(f"Probes to evaluate: {len(probes_with_gallery_embedding)}")
if probes_with_gallery_embedding.empty:
    raise RuntimeError("No probes can be evaluated after building gallery embeddings.")

results_df = evaluate_closed_set(probe_df, gallery_embeddings, model)
results_path = RESULTS_DIR / "droneface_results.csv"
results_df = pd.read_csv(results_path)
display(results_df.head())
print(f"Resultados guardados en: {results_path}")


## 10. Resultados globales


In [ ]:
gallery_embeddings = load_gallery_npz(EMBEDDINGS_DIR / "droneface_gallery_embeddings.npz")
results_df = pd.read_csv(RESULTS_DIR / "droneface_results.csv")
summary_global_df = compute_global_metrics(results_df, gallery_embeddings=gallery_embeddings, dataset_name="DroneFace")
summary_global_path = TABLES_DIR / "summary_global.csv"
summary_global_df.to_csv(summary_global_path, index=False)
display(summary_global_df)
print(f"Tabla guardada en: {summary_global_path}")


## 11. Resultados por altura


In [ ]:
droneface_by_height = compute_grouped_metrics(results_df, ["height_m"])
by_height_path = TABLES_DIR / "droneface_by_height.csv"
droneface_by_height.to_csv(by_height_path, index=False)
display(droneface_by_height)

height_fig_path = FIGS_DIR / "metrics" / "droneface_accuracy_by_height.png"
plot_metric_bar(
    droneface_by_height,
    "height_m",
    "accuracy_global",
    "DroneFace - accuracy_global por altura",
    "accuracy_global",
    height_fig_path,
)
print(f"Tabla guardada en: {by_height_path}")
print(f"Figura generada: {height_fig_path}")


## 12. Resultados por distancia


In [ ]:
droneface_by_distance = compute_grouped_metrics(results_df, ["distance_m"])
by_distance_path = TABLES_DIR / "droneface_by_distance.csv"
droneface_by_distance.to_csv(by_distance_path, index=False)
display(droneface_by_distance)

distance_fig_path = FIGS_DIR / "metrics" / "droneface_accuracy_by_distance.png"
plot_metric_bar(
    droneface_by_distance,
    "distance_m",
    "accuracy_global",
    "DroneFace - accuracy_global por distancia",
    "accuracy_global",
    distance_fig_path,
)
print(f"Tabla guardada en: {by_distance_path}")
print(f"Figura generada: {distance_fig_path}")


## 13. Resultados por altura y distancia


In [ ]:
droneface_by_height_distance = compute_grouped_metrics(results_df, ["height_m", "distance_m"])
by_height_distance_path = TABLES_DIR / "droneface_by_height_distance.csv"
droneface_by_height_distance.to_csv(by_height_distance_path, index=False)
display(droneface_by_height_distance.head(20))

heatmap_path = FIGS_DIR / "metrics" / "droneface_accuracy_height_distance.png"
plot_accuracy_heatmap_height_distance(droneface_by_height_distance, heatmap_path)

summary_fig_path = FIGS_DIR / "metrics" / "droneface_detection_vs_accuracy.png"
plot_detection_vs_accuracy(summary_global_df, summary_fig_path)

print(f"Tabla guardada en: {by_height_distance_path}")
print(f"Figura heatmap generada: {heatmap_path}")
print(f"Figura resumen generada: {summary_fig_path}")


## 14. Visualizacion de casos correctos y fallidos


In [ ]:
recognition_fig_path = FIGS_DIR / "recognition" / "droneface_correct_incorrect.png"
plot_recognition_examples(results_df, recognition_fig_path, max_correct=4, max_incorrect=4)
print(f"Figura generada: {recognition_fig_path}")


## 15. Matriz de confusion


In [ ]:
confusion_fig_path = FIGS_DIR / "metrics" / "droneface_confusion_matrix.png"
plot_confusion_matrix(results_df, confusion_fig_path)
print(f"Figura generada: {confusion_fig_path}")


## 16. Discusion preliminar automatica basada unicamente en resultados calculados


In [ ]:
discussion = generate_preliminary_discussion(
    summary_global_df,
    droneface_by_height,
    droneface_by_distance,
    droneface_by_height_distance,
)
discussion_path = TABLES_DIR / "preliminary_discussion.txt"
discussion_path.write_text(discussion, encoding="utf-8")
print(f"Discusion guardada en: {discussion_path}")
display(Markdown(discussion))


## 17. Limitaciones y trabajo futuro

Limitaciones:

- Solo se usa DroneFace porque SCface no esta disponible.
- El modelo es preentrenado y no fue ajustado al dataset.
- El experimento es closed-set, por lo que no evalua sujetos desconocidos.
- El rendimiento depende de la deteccion facial previa.
- `portraits` y `photos_all_faces` representan un baseline simplificado.
- `photos_all`, `trio_cam` y `trio_gp` no se usan aun en la evaluacion principal.
- No se implementa control de dron.
- No se evalua tiempo real.
- El uso de modelos preentrenados debe mantenerse en contexto academico/no comercial y respetar licencias.

Trabajo futuro:

- Incorporar `photos_all` con deteccion sobre imagenes completas.
- Evaluar imagenes con multiples personas de `trio_cam` y `trio_gp`.
- Probar open-set recognition con umbral de desconocido.
- Comparar ArcFace con otro metodo.
- Analizar sensibilidad a resolucion, blur, altura y distancia.
- Estudiar factibilidad de procesamiento en tiempo real.
- Eventualmente conectar con un sistema de captura o seguimiento, pero no en este avance.
